#Loading Data

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import random
import time
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn import metrics
from scipy.stats import ks_2samp
!pip install scikit-plot -q
# Correção para o erro de importação do scikit-plot com versões recentes do scipy
import scipy
scipy.interp = np.interp
import scikitplot as skplt
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# 1. Conecta o seu Google Drive ao Colab
drive.mount('/content/drive')

# 2. Define os caminhos (Agora focados no Gradient Boosting)
caminho_base = "/content/drive/MyDrive/Projeto_Churn_RN"
caminho_csv = os.path.join(caminho_base, "datasets", "customer_churn_telecom_services.csv")
caminho_modelos_gb = os.path.join(caminho_base, "modelos/GB_Entrega_V2")
caminho_logs_gb = os.path.join(caminho_base, "logs/GB_Entrega_V2")


pastas = [
    "",
    "datasets",
    "modelos", "modelos/GB_Entrega_V2",  # Pasta isolada para os arquivos do Gradient Boosting
    "logs", "logs/GB_Entrega_V2"         # Pasta isolada para os logs do Gradient Boosting
]

for pasta in pastas:
    os.makedirs(os.path.join(caminho_base, pasta), exist_ok=True)

# 3. Carrega o Dataset do Drive
df = pd.read_csv(caminho_csv)
print("Dataset carregado com sucesso!")

#Pre Processing

In [ ]:
# 0. Remoção de registros duplicados exatos (keep='first' mantém as ocorrências originais - idêntico à MLP_Entrega)
qtd_duplicatas = df.duplicated().sum()
print(f"Linhas antes da remoção de duplicatas: {len(df)} | Duplicatas exatas encontradas: {qtd_duplicatas}")
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print(f"Linhas após manter apenas as ocorrências originais: {len(df)}\n")

# 1. Correção de tipos e Nulos
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0) # Clientes no mês 0
# df = df[df['tenure'] > 0].copy().reset_index(drop=True)

# 2. Identificando Colunas para o Pipeline
colunas_categoricas_base = df.select_dtypes(include=['object']).columns.tolist()
colunas_categoricas_base.append('SeniorCitizen')

colunas_numericas_base = df.select_dtypes(include=['int16', 'int32', 'int64', 'float16', 'float32', 'float64']).columns.tolist()
colunas_numericas_base.remove('SeniorCitizen')

# 3. CONGELANDO A BASE VIRGEM PARA ABLAÇÃO
df_base = df.copy()

print(f"Base congelada para os testes. Formato: {df_base.shape}")

#Seed

In [ ]:
def fixar_seed(seed=42):
    """Garante reprodutibilidade total nos testes do Scikit-Learn."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

fixar_seed(42)
print("Sementes determinísticas fixadas!")

#Split

In [ ]:
def realizar_split(df_input):
    """Separação 50/25/25 estratificada."""
    df_class0 = df_input[df_input['Churn'] == 'No']
    df_class1 = df_input[df_input['Churn'] == 'Yes']

    train_0, temp_0 = train_test_split(df_class0, test_size=0.5, random_state=42)
    val_0, test_0 = train_test_split(temp_0, test_size=0.5, random_state=42)

    train_1, temp_1 = train_test_split(df_class1, test_size=0.5, random_state=42)
    val_1, test_1 = train_test_split(temp_1, test_size=0.5, random_state=42)

    df_train_out = shuffle(pd.concat([train_0, train_1]), random_state=42)
    df_val_out = shuffle(pd.concat([val_0, val_1]), random_state=42)
    df_test_out = shuffle(pd.concat([test_0, test_1]), random_state=42)
    return df_train_out, df_val_out, df_test_out

def aplicar_pipeline_e_oversampling_gb(df_train_in, df_val_in, df_test_in, cols_cat, cols_num):
    # 1. Separar Variáveis Independentes (X) do Alvo (y)
    X_tr = df_train_in.drop('Churn', axis=1)
    y_tr = df_train_in['Churn'].map({'No': 0, 'Yes': 1})
    X_v = df_val_in.drop('Churn', axis=1)
    y_v = df_val_in['Churn'].map({'No': 0, 'Yes': 1})
    X_te = df_test_in.drop('Churn', axis=1)
    y_te = df_test_in['Churn'].map({'No': 0, 'Yes': 1})

    cat_c = [col for col in cols_cat if col != 'Churn']

    # 2. PIPELINE ENXUTO: Apenas One-Hot Encoding! O resto passa direto.
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(drop='if_binary', sparse_output=False), cat_c)
        ],
        remainder='passthrough', # As colunas numéricas (sem ser categóricas) passam sem alteração
        verbose_feature_names_out=False
    )

    X_tr_enc = preprocessor.fit_transform(X_tr)
    X_v_enc = preprocessor.transform(X_v)
    X_te_enc = preprocessor.transform(X_te)

    nomes_colunas = preprocessor.get_feature_names_out()

    X_tr_df = pd.DataFrame(X_tr_enc, columns=nomes_colunas, index=X_tr.index)
    X_v_df = pd.DataFrame(X_v_enc, columns=nomes_colunas, index=X_v.index)
    X_te_df = pd.DataFrame(X_te_enc, columns=nomes_colunas, index=X_te.index)

    # 3. Oversampling Manual (Mantido 100% igual à MLP)
    train_full = pd.concat([X_tr_df, y_tr], axis=1)
    val_full = pd.concat([X_v_df, y_v], axis=1)

    def balancear(df_dados):
        df_0 = df_dados[df_dados['Churn'] == 0]
        df_1 = df_dados[df_dados['Churn'] == 1]
        diff = len(df_0) - len(df_1)
        if diff > 0:
            df_1_over = df_1.sample(n=diff, replace=True, random_state=42)
            return shuffle(pd.concat([df_0, df_1, df_1_over]), random_state=42)
        return df_dados

    train_bal = balancear(train_full)
    val_bal = balancear(val_full)

    X_tr_final = train_bal.drop('Churn', axis=1)
    y_tr_final = train_bal['Churn']
    X_v_final = val_bal.drop('Churn', axis=1)
    y_v_final = val_bal['Churn']

    return X_tr_final, y_tr_final, X_v_final, y_v_final, X_te_df, y_te

# Executa o processo na base limpa para termos o dataset de "Base" do Gradient Boosting
df_train_base, df_val_base, df_test_base = realizar_split(df_base)
X_train_gb, y_train_gb, X_val_gb, y_val_gb, X_test_gb, y_test_gb = aplicar_pipeline_e_oversampling_gb(
    df_train_base, df_val_base, df_test_base, colunas_categoricas_base, colunas_numericas_base
)

print(f"Pipeline do Gradient Boosting pronto! Tamanho das features: {X_train_gb.shape}")

#Get Scores Function

In [ ]:
# Correção para o erro de importação do scikit-plot com versões recentes do scipy
import scipy
import numpy as np
scipy.interp = np.interp

from sklearn import metrics
import scikitplot as skplt
from scipy.stats import ks_2samp

def get_scores_churn(targets, probs_1, threshold=0.5):
    """Calcula as métricas para um dado threshold, e extrai o threshold ótimo do KS"""
    predictions = (probs_1 >= threshold).astype(int)

    probs_0_class = probs_1[targets == 0]
    probs_1_class = probs_1[targets == 1]

    if len(probs_0_class) > 0 and len(probs_1_class) > 0:
        ks_stat, _ = ks_2samp(probs_0_class, probs_1_class)
    else:
        ks_stat = 0.0

    if len(np.unique(targets)) > 1:
        fpr, tpr, thresholds = metrics.roc_curve(targets, probs_1)
        optimal_idx = np.argmax(tpr - fpr)
        optimal_threshold = thresholds[optimal_idx]
    else:
        optimal_threshold = 0.5

    probs_clipped = np.clip(probs_1, 1e-7, 1 - 1e-7)
    mse = metrics.mean_squared_error(targets, probs_1)
    log_loss_val = metrics.log_loss(targets, probs_clipped) if len(np.unique(targets)) > 1 else 0.0

    return {
        "accuracy": metrics.accuracy_score(targets, predictions),
        "precision": metrics.precision_score(targets, predictions, zero_division=0),
        "recall": metrics.recall_score(targets, predictions, zero_division=0),
        "f1_score": metrics.f1_score(targets, predictions, zero_division=0),
        "roc_auc": metrics.roc_auc_score(targets, probs_1) if len(np.unique(targets)) > 1 else 0.0,
        "ks_stat": ks_stat,
        "mse": mse,
        "log_loss": log_loss_val,
        "optimal_threshold": optimal_threshold
    }

#Training Function

In [ ]:
import copy
import joblib
import time

def executar_experimento_gb(name_experiment, learning_rate, n_estimators, max_depth,
                            min_samples_split, min_samples_leaf, subsample,
                            X_train, y_train, X_val, y_val, max_features=None, patience=60):
    start_time = time.time()
    fixar_seed(42)
    print(f"\n{'-'*120}")
    print(f"EXPERIMENTO GB: {name_experiment} | LR: {learning_rate} | Est: {n_estimators} | Depth: {max_depth} | Split: {min_samples_split} | Leaf: {min_samples_leaf} | Sub: {subsample} | MaxFeat: {max_features}")

    # Treinamento incremental com warm_start=True para Early Stopping na validação EXTERNA (sem vazamento interno)
    modelo_gb = GradientBoostingClassifier(
        learning_rate=learning_rate,
        n_estimators=1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        subsample=subsample,
        max_features=max_features,
        warm_start=True,
        random_state=42
    )

    best_val_loss = float('inf')
    best_model = None
    best_iter = 0
    patience_counter = 0

    for i in range(1, n_estimators + 1):
        modelo_gb.set_params(n_estimators=i)
        modelo_gb.fit(X_train, y_train)

        probs_val_i = modelo_gb.predict_proba(X_val)[:, 1]
        val_loss_i = metrics.log_loss(y_val, np.clip(probs_val_i, 1e-7, 1 - 1e-7))

        if val_loss_i < best_val_loss:
            best_val_loss = val_loss_i
            best_iter = i
            best_model = copy.deepcopy(modelo_gb)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Avaliação final usando exatamente o modelo da melhor iteração de val_loss
    probs_train = best_model.predict_proba(X_train)[:, 1]
    probs_val = best_model.predict_proba(X_val)[:, 1]

    scores_train = get_scores_churn(np.array(y_train), probs_train)
    scores = get_scores_churn(np.array(y_val), probs_val)

    val_ks = scores['ks_stat']
    val_acc = scores['accuracy']
    val_prec = scores['precision']
    val_rec = scores['recall']
    val_f1 = scores['f1_score']
    val_auc = scores['roc_auc']
    val_mse = scores['mse']
    train_mse = scores_train['mse']

    caminho_arquivo = os.path.join(caminho_modelos_gb, f'{name_experiment}.joblib')
    joblib.dump(best_model, caminho_arquivo)

    end_time = time.time()
    tempo_execucao = end_time - start_time

    print(f"Resultados: Val Loss: {best_val_loss:.4f} | KS: {val_ks:.4f} | AUC: {val_auc:.4f} | Train MSE: {train_mse:.4f} | Val MSE: {val_mse:.4f} | F1: {val_f1:.4f} | Tempo: {tempo_execucao:.2f}s")
    print(f"Árvores efetivamente mantidas (Early Stopping Externo): {best_iter} / {n_estimators}")

    return {
        'nome': name_experiment,
        'val_loss': best_val_loss,
        'val_ks': val_ks,
        'val_auc': val_auc,
        'train_mse': train_mse,
        'val_mse': val_mse,
        'val_acc': val_acc,
        'val_prec': val_prec,
        'val_rec': val_rec,
        'val_f1': val_f1,
        'best_n_estimators': best_iter,
        'learning_rate': learning_rate,
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'subsample': subsample,
        'max_features': max_features,
        'tempo_execucao': tempo_execucao
    }

# Winner Function

In [ ]:
from sklearn.metrics import RocCurveDisplay, confusion_matrix, classification_report

def plotar_vencedor_gb(resultados_do_bloco, X_val, y_val):
    # Elege o vencedor baseado no MAIOR KS
    vencedor = max(resultados_do_bloco, key=lambda x: x['val_ks'])
    print(f"\n{'='*90}")
    print(f"VENCEDOR DA BATERIA: {vencedor['nome']}")
    print(f"Métricas: KS: {vencedor['val_ks']:.4f} | AUC: {vencedor.get('val_auc', 0):.4f} | Acc: {vencedor.get('val_acc', 0):.4f} | Prec: {vencedor.get('val_prec', 0):.4f} | Rec: {vencedor.get('val_rec', 0):.4f} | F1: {vencedor.get('val_f1', 0):.4f}")
    print(f"{'='*90}")

    # Carrega o melhor modelo salvo
    caminho_arquivo = os.path.join(caminho_modelos_gb, f"{vencedor['nome']}.joblib")
    modelo = joblib.load(caminho_arquivo)

    # Previsões
    probs_1 = modelo.predict_proba(X_val)[:, 1]
    y_true = np.array(y_val)

    # Obtém scores e o limiar ótimo
    scores_default = get_scores_churn(y_true, probs_1, threshold=0.5)
    optimal_threshold = scores_default['optimal_threshold']
    scores_optimal = get_scores_churn(y_true, probs_1, threshold=optimal_threshold)

    # Formatação para o scikit-plot [Prob Classe 0, Prob Classe 1]
    probs_0 = 1 - probs_1
    y_pred_scores = np.column_stack((probs_0, probs_1))

    y_pred_class_default = (probs_1 >= 0.5).astype(int)
    y_pred_class_optimal = (probs_1 >= optimal_threshold).astype(int)

    # Plotagens (4 subplots)
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    skplt.metrics.plot_ks_statistic(y_true, y_pred_scores, ax=axes[0], title=f"KS: {vencedor['nome']}")

    RocCurveDisplay.from_predictions(y_true, probs_1, ax=axes[1], name=vencedor['nome'])
    axes[1].set_title(f"Curva ROC: {vencedor['nome']}")
    axes[1].plot([0, 1], [0, 1], 'k--', label='Aleatório')
    axes[1].legend()

    cm_default = confusion_matrix(y_true, y_pred_class_default, normalize='true')
    sns.heatmap(cm_default, annot=True, fmt=".2%", cmap="Blues", ax=axes[2], xticklabels=['Não Churn (0)', 'Churn (1)'], yticklabels=['Não Churn (0)', 'Churn (1)'])
    axes[2].set_title("Matriz (Thr: 0.500)")
    axes[2].set_ylabel("Valor Real")
    axes[2].set_xlabel("Predição")

    cm_optimal = confusion_matrix(y_true, y_pred_class_optimal, normalize='true')
    sns.heatmap(cm_optimal, annot=True, fmt=".2%", cmap="Greens", ax=axes[3], xticklabels=['Não Churn (0)', 'Churn (1)'], yticklabels=['Não Churn (0)', 'Churn (1)'])
    axes[3].set_title(f"Matriz (Thr KS: {optimal_threshold:.3f})")
    axes[3].set_ylabel("Valor Real")
    axes[3].set_xlabel("Predição")

    plt.tight_layout()
    plt.show()

    # --- NOVO: Print dos Relatórios Textuais (Idêntico ao MLP) ---
    print(f"--- MÉTRICAS COM THRESHOLD PADRÃO (0.500) ---")
    print(f"KS: {scores_default['ks_stat']:.4f} | AUROC: {scores_default['roc_auc']:.4f} | MSE: {scores_default['mse']:.4f} | Log-Loss: {scores_default['log_loss']:.4f} | Acurácia: {scores_default['accuracy']:.4f}")
    print(f"Precisão: {scores_default['precision']:.4f} | Recall: {scores_default['recall']:.4f} | F1-Score: {scores_default['f1_score']:.4f}\n")
    print("Relatório de Classificação (Threshold Padrão):")
    print(classification_report(y_true, y_pred_class_default, target_names=['Não Churn (0)', 'Churn (1)']))

    print(f"\n--- MÉTRICAS COM THRESHOLD OTIMIZADO PELO KS ({optimal_threshold:.3f}) ---")
    print(f"KS: {scores_optimal['ks_stat']:.4f} | AUROC: {scores_optimal['roc_auc']:.4f} | Acurácia: {scores_optimal['accuracy']:.4f}")
    print(f"Precisão: {scores_optimal['precision']:.4f} | Recall: {scores_optimal['recall']:.4f} | F1-Score: {scores_optimal['f1_score']:.4f}\n")
    print("Relatório de Classificação (Threshold Otimizado):")
    print(classification_report(y_true, y_pred_class_optimal, target_names=['Não Churn (0)', 'Churn (1)']))

    return vencedor

#Plotando Evolução Parâmetros

In [ ]:
def plotar_evolucao_parametros(df_resultados, parametro_x):
    """Plota evolução de KS (Validação) e MSE (Treino vs Validação) + tabela resumo para cada bateria."""
    df_plot = df_resultados.copy()
    df_plot[f'{parametro_x}_str'] = df_plot[parametro_x].astype(str)

    tem_mse = 'train_mse' in df_plot.columns and 'val_mse' in df_plot.columns

    # Tabela resumo da bateria
    cols_tabela = [parametro_x, 'val_ks']
    if 'val_loss' in df_plot.columns:
        cols_tabela.append('val_loss')
    if tem_mse:
        cols_tabela = [parametro_x, 'val_ks', 'train_mse', 'val_mse', 'val_loss']
    display(df_plot[cols_tabela].round(4))

    if tem_mse:
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))

        # --- Gráfico 1: KS (Validação) ---
        max_ks = df_plot['val_ks'].max()
        cores = ['green' if ks == max_ks else 'steelblue' for ks in df_plot['val_ks']]
        sns.barplot(data=df_plot, x=f'{parametro_x}_str', y='val_ks', hue=f'{parametro_x}_str',
                    palette=cores, legend=False, ax=axes[0])
        for p in axes[0].patches:
            if p.get_height() > 0:
                axes[0].annotate(format(p.get_height(), '.4f'),
                                 (p.get_x() + p.get_width() / 2., p.get_height()),
                                 ha='center', va='center', xytext=(0, 8), textcoords='offset points', fontsize=9)
        axes[0].set_ylim(0, max_ks * 1.18)
        axes[0].set_title(f'Comparativo de KS por {parametro_x.replace("_", " ").title()}', fontsize=13)
        axes[0].set_xlabel(parametro_x.replace('_', ' ').title(), fontsize=11)
        axes[0].set_ylabel('KS (Validação)', fontsize=11, fontweight='bold')
        axes[0].grid(axis='y', linestyle='--', alpha=0.6)
        axes[0].tick_params(axis='x', rotation=45)

        # --- Gráfico 2: MSE (Treino vs Validação) ---
        df_mse = df_plot[[f'{parametro_x}_str', 'train_mse', 'val_mse']].melt(
            id_vars=f'{parametro_x}_str',
            value_vars=['train_mse', 'val_mse'],
            var_name='Conjunto',
            value_name='MSE'
        )
        df_mse['Conjunto'] = df_mse['Conjunto'].map({'train_mse': 'Treino', 'val_mse': 'Validação'})
        sns.barplot(data=df_mse, x=f'{parametro_x}_str', y='MSE', hue='Conjunto',
                    palette={'Treino': '#4C72B0', 'Validação': '#DD8452'}, ax=axes[1])
        for p in axes[1].patches:
            if p.get_height() > 0:
                axes[1].annotate(format(p.get_height(), '.4f'),
                                 (p.get_x() + p.get_width() / 2., p.get_height()),
                                 ha='center', va='center', xytext=(0, 8), textcoords='offset points', fontsize=8)
        axes[1].set_ylim(0, df_mse['MSE'].max() * 1.18)
        axes[1].set_title(f'MSE (Treino vs Validação) por {parametro_x.replace("_", " ").title()}', fontsize=13)
        axes[1].set_xlabel(parametro_x.replace('_', ' ').title(), fontsize=11)
        axes[1].set_ylabel('MSE (Menor é Melhor)', fontsize=11, fontweight='bold')
        axes[1].grid(axis='y', linestyle='--', alpha=0.6)
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].legend(title='Conjunto')

        plt.tight_layout()
        plt.show()
    else:
        plt.figure(figsize=(10, 5))
        max_ks = df_plot['val_ks'].max()
        cores = ['green' if ks == max_ks else 'steelblue' for ks in df_plot['val_ks']]
        ax = sns.barplot(data=df_plot, x=f'{parametro_x}_str', y='val_ks', hue=f'{parametro_x}_str',
                         palette=cores, legend=False)
        for p in ax.patches:
            if p.get_height() > 0:
                ax.annotate(format(p.get_height(), '.4f'),
                            (p.get_x() + p.get_width() / 2., p.get_height()),
                            ha='center', va='center', xytext=(0, 9), textcoords='offset points')
        plt.ylim(0, max_ks * 1.18)
        plt.title(f'Comparativo de KS por {parametro_x.replace("_", " ").title()}', fontsize=14)
        plt.xlabel(parametro_x.replace('_', ' ').title(), fontsize=12)
        plt.ylabel('Melhor KS', fontsize=12, fontweight='bold')
        plt.grid(axis='y', linestyle='--', alpha=0.6)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

### Bateria 1: Testando a Quantidade de Árvores (`n_estimators`)
Vamos testar diferentes números de estimadores mantendo a taxa de aprendizado e a profundidade fixas.

In [ ]:
print("Iniciando Bateria 1: n_estimators...")
resultados_b1 = []
valores_n_est = [100, 200, 300, 500, 1000, 1500, 2000, 2500, 3000]

for n in valores_n_est:
    res = executar_experimento_gb(
        name_experiment=f"GB_B1_nest_{n}",
        learning_rate=0.1,
        n_estimators=n,
        max_depth=3,
        min_samples_split=2,
        min_samples_leaf=1,
        subsample=1.0,
        X_train=X_train_gb,
        y_train=y_train_gb,
        X_val=X_val_gb,
        y_val=y_val_gb
    )
    resultados_b1.append(res)

import pandas as pd
import os

# 1. Salva o log no Drive para proteger os dados (Como no MLP)
df_resultados_b1 = pd.DataFrame(resultados_b1)
df_resultados_b1.to_csv(os.path.join(caminho_logs_gb, "bateria_1_n_estimators.csv"), index=False)

# 2. Mostra o vencedor e os 4 gráficos (KS, ROC, Matrizes) e Relatórios
vencedor_b1 = plotar_vencedor_gb(resultados_b1, X_val_gb, y_val_gb)

# 3. Plota o gráfico de barras da evolução do KS na bateria (Como no MLP)
plotar_evolucao_parametros(df_resultados_b1, 'n_estimators')

### Bateria 2: Testando a Taxa de Aprendizado (`learning_rate`)
Herdando o melhor número de estimadores da Bateria 1, testamos o peso/passo de cada árvore.

In [ ]:
print("Iniciando Bateria 2: learning_rate...")
resultados_b2 = []
valores_lr = [0.01, 0.05, 0.1, 0.2, 0.001, 0.005, 0.9]
melhor_n_est = vencedor_b1['n_estimators']

for lr in valores_lr:
    res = executar_experimento_gb(
        name_experiment=f"GB_B2_lr_{lr}",
        learning_rate=lr,
        n_estimators=melhor_n_est,
        max_depth=3,
        min_samples_split=2,
        min_samples_leaf=1,
        subsample=1.0,
        X_train=X_train_gb,
        y_train=y_train_gb,
        X_val=X_val_gb,
        y_val=y_val_gb
    )
    resultados_b2.append(res)

import pandas as pd
import os

# 1. Salva o log no Drive para proteger os dados (Como no MLP)
df_resultados_b2 = pd.DataFrame(resultados_b2)
df_resultados_b2.to_csv(os.path.join(caminho_logs_gb, "bateria_2_learning_rate.csv"), index=False)

# 2. Mostra o vencedor e os 4 gráficos (KS, ROC, Matrizes) e Relatórios
vencedor_b2 = plotar_vencedor_gb(resultados_b2, X_val_gb, y_val_gb)

# 3. Plota o gráfico de barras da evolução do KS na bateria (Como no MLP)
plotar_evolucao_parametros(df_resultados_b2, 'learning_rate')

### Bateria 3: Testando a Profundidade (`max_depth`)
Com o melhor número de árvores e a melhor taxa de aprendizado definidos, avaliamos a complexidade estrutural (se a árvore será rasa ou profunda).

In [ ]:
print("Iniciando Bateria 3: max_depth...")
resultados_b3 = []
valores_depth = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
melhor_lr = vencedor_b2['learning_rate']

for d in valores_depth:
    res = executar_experimento_gb(
        name_experiment=f"GB_B3_depth_{d}",
        learning_rate=melhor_lr,
        n_estimators=melhor_n_est,
        max_depth=d,
        min_samples_split=2,
        min_samples_leaf=1,
        subsample=1.0,
        X_train=X_train_gb,
        y_train=y_train_gb,
        X_val=X_val_gb,
        y_val=y_val_gb
    )
    resultados_b3.append(res)

import pandas as pd
import os

# 1. Salva o log no Drive para proteger os dados (Como no MLP)
df_resultados_b3 = pd.DataFrame(resultados_b3)
df_resultados_b3.to_csv(os.path.join(caminho_logs_gb, "bateria_3_max_depth.csv"), index=False)

# 2. Mostra o vencedor e os 4 gráficos (KS, ROC, Matrizes) e Relatórios
vencedor_b3 = plotar_vencedor_gb(resultados_b3, X_val_gb, y_val_gb)

# 3. Plota o gráfico de barras da evolução do KS na bateria (Como no MLP)
plotar_evolucao_parametros(df_resultados_b3, 'max_depth')

### Bateria 4: Testando Amostragem (`subsample`)
Por fim, testamos se usar apenas uma fração dos dados para treinar cada árvore (bagging) ajuda na regularização e no KS final.

In [ ]:
print("Iniciando Bateria 4: subsample...")
resultados_b4 = []
valores_sub = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
melhor_depth = vencedor_b3['max_depth']

for sub in valores_sub:
    res = executar_experimento_gb(
        name_experiment=f"GB_B4_sub_{sub}",
        learning_rate=melhor_lr,
        n_estimators=melhor_n_est,
        max_depth=melhor_depth,
        min_samples_split=2,
        min_samples_leaf=1,
        subsample=sub,
        X_train=X_train_gb,
        y_train=y_train_gb,
        X_val=X_val_gb,
        y_val=y_val_gb
    )
    resultados_b4.append(res)

import pandas as pd
import os

# 1. Salva o log no Drive para proteger os dados (Como no MLP)
df_resultados_b4 = pd.DataFrame(resultados_b4)
df_resultados_b4.to_csv(os.path.join(caminho_logs_gb, "bateria_4_subsample.csv"), index=False)

# 2. Mostra o vencedor e os 4 gráficos (KS, ROC, Matrizes) e Relatórios
vencedor_b4 = plotar_vencedor_gb(resultados_b4, X_val_gb, y_val_gb)

# 3. Plota o gráfico de barras da evolução do KS na bateria (Como no MLP)
plotar_evolucao_parametros(df_resultados_b4, 'subsample')

### Bateria 5: Testando Amostras Mínimas para Divisão (`min_samples_split`)
Define o número mínimo de amostras exigidas para dividir um nó interno. Valores maiores previnem que o modelo aprenda relações muito específicas (overfitting).

In [ ]:
print("Iniciando Bateria 5: min_samples_split...")
resultados_b5 = []
valores_split = [2, 5, 10, 20, 50, 100]

# Recuperando os melhores parâmetros consolidados até a bateria 4
melhor_lr = vencedor_b4['learning_rate']
melhor_n_est = vencedor_b4['n_estimators']
melhor_depth = vencedor_b4['max_depth']
melhor_sub = vencedor_b4['subsample']

for split in valores_split:
    res = executar_experimento_gb(
        name_experiment=f"GB_B5_split_{split}",
        learning_rate=melhor_lr,
        n_estimators=melhor_n_est,
        max_depth=melhor_depth,
        min_samples_split=split,
        min_samples_leaf=1,
        subsample=melhor_sub,
        X_train=X_train_gb,
        y_train=y_train_gb,
        X_val=X_val_gb,
        y_val=y_val_gb
    )
    resultados_b5.append(res)

import pandas as pd
import os

# 1. Salva o log no Drive para proteger os dados (Como no MLP)
df_resultados_b5 = pd.DataFrame(resultados_b5)
df_resultados_b5.to_csv(os.path.join(caminho_logs_gb, "bateria_5_min_samples_split.csv"), index=False)

# 2. Mostra o vencedor e os 4 gráficos (KS, ROC, Matrizes) e Relatórios
vencedor_b5 = plotar_vencedor_gb(resultados_b5, X_val_gb, y_val_gb)

# 3. Plota o gráfico de barras da evolução do KS na bateria (Como no MLP)
plotar_evolucao_parametros(df_resultados_b5, 'min_samples_split')

### Bateria 6: Testando Seleção de Colunas por Árvore (`max_features` / `colsample_bytree`)
Testaremos a utilização de apenas um subconjunto de colunas em cada quebra de árvore. Pode ser uma fração das colunas ou funções matemáticas (`sqrt`, `log2`).

In [ ]:
print("Iniciando Bateria 6: max_features (colsample_bytree)...")
resultados_b6 = []
valores_features = [None, 'sqrt', 'log2', 0.5, 0.6, 0.7, 0.8, 0.9, 1]

melhor_split = vencedor_b5['min_samples_split']

for feat in valores_features:
    feat_str = str(feat).replace('.', '') if feat is not None else "None"
    res = executar_experimento_gb(
        name_experiment=f"GB_B6_colsample_{feat_str}",
        learning_rate=melhor_lr,
        n_estimators=melhor_n_est,
        max_depth=melhor_depth,
        min_samples_split=melhor_split,
        min_samples_leaf=1,
        subsample=melhor_sub,
        max_features=feat,
        X_train=X_train_gb,
        y_train=y_train_gb,
        X_val=X_val_gb,
        y_val=y_val_gb
    )
    resultados_b6.append(res)

import pandas as pd
import os

# 1. Salva o log no Drive para proteger os dados (Como no MLP)
df_resultados_b6 = pd.DataFrame(resultados_b6)
df_resultados_b6.to_csv(os.path.join(caminho_logs_gb, "bateria_6_max_features.csv"), index=False)

# 2. Mostra o vencedor e os 4 gráficos (KS, ROC, Matrizes) e Relatórios
vencedor_b6 = plotar_vencedor_gb(resultados_b6, X_val_gb, y_val_gb)

# 3. Plota o gráfico de barras da evolução do KS na bateria (Como no MLP)
plotar_evolucao_parametros(df_resultados_b6, 'max_features')

## Optuna: Otimização Global de Hiperparâmetros (Baseline Original)
Agora que entendemos os limites de cada parâmetro isoladamente, vamos usar o Optuna para encontrar a melhor combinação cruzada entre eles.

In [ ]:
!pip install optuna -q

###Objective Function

In [ ]:
import optuna
from optuna.samplers import TPESampler
import os

def objective_gb(trial):
    # 1. Sugestões de hiperparâmetros baseadas nas baterias manuais
    # Espaço regularizado contra memorização das cópias do oversampling
    lr = trial.suggest_float('learning_rate', 0.005, 0.15, log=True)
    n_est = trial.suggest_int('n_estimators', 100, 3000)
    depth = trial.suggest_int('max_depth', 1, 4)
    split = trial.suggest_int('min_samples_split', 2, 100)
    leaf = trial.suggest_int('min_samples_leaf', 15, 80)
    sub = trial.suggest_float('subsample', 0.5, 1.0)
    max_feat = trial.suggest_categorical('max_features', [None, 'sqrt', 'log2', 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])

    nome_trial = f"Optuna_Trial_{trial.number}"

    # 2. Roda o experimento (já salva o .joblib de cada trial no Drive!)
    resultado = executar_experimento_gb(
        name_experiment=nome_trial,
        learning_rate=lr,
        n_estimators=n_est,
        max_depth=depth,
        min_samples_split=split,
        min_samples_leaf=leaf,
        subsample=sub,
        max_features=max_feat,
        X_train=X_train_gb,
        y_train=y_train_gb,
        X_val=X_val_gb,
        y_val=y_val_gb
    )

    # 3. Retorna o KS
    return resultado['val_ks']

###Run and Save

In [ ]:
caminho_banco_optuna = f"sqlite:///{caminho_logs_gb}/optuna_gb_baseline.db"

RECOMECAR_DO_ZERO = False # False garante retomada automática sem apagar trials já salvos

if RECOMECAR_DO_ZERO:
    try:
        optuna.delete_study(study_name="Otimizacao_GB", storage=caminho_banco_optuna)
        print("Estudo anterior apagado do banco de dados. Começando do zero!")
    except KeyError:
        pass

sampler = TPESampler(seed=42)

study_gb = optuna.create_study(
    study_name="Otimizacao_GB",
    direction="maximize",
    storage=caminho_banco_optuna,
    sampler=sampler,
    load_if_exists=True
)

n_trials_restantes = max(0, 100 - len(study_gb.trials))
print(f"Iniciando/Retomando Optuna GB (trials concluídos: {len(study_gb.trials)} | restantes: {n_trials_restantes})...")
if n_trials_restantes > 0:
    study_gb.optimize(objective_gb, n_trials=n_trials_restantes)

print("\n=== MELHOR MODELO ENCONTRADO PELO OPTUNA ===")
print(f"Melhor KS: {study_gb.best_value:.4f}")
print("Melhores Parâmetros:")
for key, value in study_gb.best_params.items():
    print(f"    {key}: {value}")

###Evaluate Best Model

In [ ]:
import optuna

# Carrega o estudo salvo no banco de dados
caminho_banco_optuna = f"sqlite:///{caminho_logs_gb}/optuna_gb_baseline.db"
study_gb_loaded = optuna.load_study(study_name="Otimizacao_GB", storage=caminho_banco_optuna)

best_trial_gb = study_gb_loaded.best_trial

# Recria o dicionário do vencedor para ser compatível com plotar_vencedor_gb
resultado_campeao_optuna_gb = {
    'nome': f"Optuna_Trial_{best_trial_gb.number}",
    'val_ks': best_trial_gb.value,
    **best_trial_gb.params
}

print("Calculando e plotando métricas para o Grande Campeão do Optuna (Base)...")
_ = plotar_vencedor_gb([resultado_campeao_optuna_gb], X_val_gb, y_val_gb)

# Feature Engineering para Churn de Telecom

##Função de Plotagem de Comparação

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_feature_engineering_comparison_gb(name_fe, ks_b6_base, ks_b6_fe, ks_optuna_base, ks_optuna_fe):
    """
    Cria um gráfico de barras comparando os resultados de KS antes e depois
    da feature engineering, para os cenários manual (B6) e Optuna.
    """
    labels = ['Manual (B6) - Sem FE', f'Manual (B6) - Com {name_fe}',
              'Optuna - Sem FE', f'Optuna - Com {name_fe}']
    ks_values = [ks_b6_base, ks_b6_fe, ks_optuna_base, ks_optuna_fe]

    cores = ['lightsteelblue', 'royalblue', 'lightgreen', 'forestgreen']

    fig, ax = plt.subplots(figsize=(10, 6))
    barras = ax.bar(labels, ks_values, color=cores)

    for bar in barras:
        height = bar.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=11)

    ax.set_ylim(min(ks_values) - 0.01, max(ks_values) + 0.01)
    ax.set_ylabel('Estatística KS', fontsize=12)
    ax.set_title(f'Impacto da Feature no Gradient Boosting: {name_fe}', fontsize=14, pad=15)

    plt.xticks(rotation=15, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

print("Função de plotagem de comparação do GB criada com sucesso!")

### Setup para os Testes de Feature Engineering (Ablation Study)
Rode esta célula **uma vez** antes de testar qualquer FE. Ela carrega os hiperparâmetros do seu melhor modelo Manual (Bateria 6) e do melhor modelo Optuna direto do Drive, para que você possa rodar os testes de FE em qualquer ordem.

In [ ]:
import pandas as pd
import numpy as np
import optuna
import os

def tratar_max_features(val):
    """Converte o valor lido do CSV de volta para o tipo aceito pelo Scikit-Learn."""
    if pd.isna(val) or str(val) == 'None':
        return None
    if str(val) in ['sqrt', 'log2']:
        return str(val)
    try:
        f_val = float(val)
        return int(f_val) if f_val == 1.0 and str(val) == '1' else f_val
    except ValueError:
        return val

# --- 1. Carregando Hiperparâmetros do Campeão Manual (Bateria 6) ---
print("Carregando parâmetros do Campeão Manual (B6)...")
caminho_csv_b6 = os.path.join(caminho_logs_gb, "bateria_6_max_features.csv")
df_b6 = pd.read_csv(caminho_csv_b6)
melhor_linha_b6 = df_b6.loc[df_b6['val_ks'].idxmax()]
vencedor_b6 = melhor_linha_b6.to_dict()
vencedor_b6['max_features'] = tratar_max_features(vencedor_b6['max_features'])
ks_b6_base = vencedor_b6['val_ks']
print(f"Sucesso! Campeão Manual ({vencedor_b6['nome']}) carregado. KS: {ks_b6_base:.4f}")

# --- 2. Carregando Hiperparâmetros do Campeão Optuna (Base) ---
print("Carregando parâmetros do Campeão Optuna Base...")
caminho_banco_optuna = f"sqlite:///{caminho_logs_gb}/optuna_gb_baseline.db"
study_gb_loaded = optuna.load_study(study_name="Otimizacao_GB", storage=caminho_banco_optuna)
best_trial_gb = study_gb_loaded.best_trial
ks_optuna_base_gb = best_trial_gb.value
print(f"Sucesso! Campeão Optuna Base (Trial {best_trial_gb.number}) carregado. KS: {ks_optuna_base_gb:.4f}")

# --- 3. Função Unificada para Rodar Optuna e Avaliar Feature Engineering ---
def rodar_optuna_e_avaliar_gb(nome_estudo, X_train, y_train, X_val, y_val, prefixo_trial="Optuna_Trial"):
    caminho_banco = f"sqlite:///{caminho_logs_gb}/{nome_estudo.lower()}.db"

    def objective(trial):
        # Espaço regularizado contra memorização das cópias do oversampling
        lr = trial.suggest_float('learning_rate', 0.005, 0.15, log=True)
        n_est = trial.suggest_int('n_estimators', 100, 3000)
        depth = trial.suggest_int('max_depth', 1, 4)
        split = trial.suggest_int('min_samples_split', 2, 100)
        leaf = trial.suggest_int('min_samples_leaf', 15, 80)
        sub = trial.suggest_float('subsample', 0.5, 1.0)
        max_feat = trial.suggest_categorical('max_features', [None, 'sqrt', 'log2', 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])

        nome_trial = f"{prefixo_trial}_{trial.number}"

        resultado = executar_experimento_gb(
            name_experiment=nome_trial,
            learning_rate=lr,
            n_estimators=n_est,
            max_depth=depth,
            min_samples_split=split,
            min_samples_leaf=leaf,
            subsample=sub,
            max_features=max_feat,
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val
        )
        return resultado['val_ks']

    RECOMECAR_DO_ZERO = False # False garante retomada automática sem apagar trials já salvos
    if RECOMECAR_DO_ZERO:
        try:
            optuna.delete_study(study_name=nome_estudo, storage=caminho_banco)
            print(f"Estudo {nome_estudo} apagado do banco de dados. Começando do zero!")
        except KeyError:
            pass

    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(
        study_name=nome_estudo,
        direction="maximize",
        storage=caminho_banco,
        sampler=sampler,
        load_if_exists=True
    )

    n_trials_restantes = max(0, 100 - len(study.trials))
    print(f"Iniciando/Retomando Optuna para: {nome_estudo} (concluídos: {len(study.trials)} | restantes: {n_trials_restantes})...")
    if n_trials_restantes > 0:
        study.optimize(objective, n_trials=n_trials_restantes)

    print(f"\n=== MELHOR MODELO ENCONTRADO ({nome_estudo}) ===")
    print(f"Melhor KS: {study.best_value:.4f}")
    print("Melhores Parâmetros:")
    for key, value in study.best_params.items():
        print(f"    {key}: {value}")

    # --- AVALIAÇÃO FINAL (VENCEDOR) ---
    melhor_trial = study.best_trial
    resultado_campeao = {
        'nome': f"{prefixo_trial}_{melhor_trial.number}",
        'val_ks': melhor_trial.value,
        **melhor_trial.params
    }

    print(f"\nCalculando e plotando métricas para o Grande Campeão ({nome_estudo})...")
    plotar_vencedor_gb([resultado_campeao], X_val, y_val)

    return study

print("\nSetup concluído! Agora você pode rodar os blocos FE_1, FE_2, FE_3, FE_4 ou FE_ALL independentemente.")

##FE_1: Qtde_Servicos

In [ ]:
# 1. Puxa a cópia da base virgem e das listas
df_fe1 = df_base.copy()
cols_cat_fe1 = colunas_categoricas_base.copy()
cols_num_fe1 = colunas_numericas_base.copy()

# 2. Aplica a transformação
colunas_servicos = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df_fe1['Qtde_Servicos'] = df_fe1[colunas_servicos].apply(lambda linha: (linha == 'Yes').sum(), axis=1)

# 3. Adiciona APENAS na lista temporária deste escopo
if 'Qtde_Servicos' not in cols_num_fe1:
    cols_num_fe1.append('Qtde_Servicos')

display(df_fe1.head())

### Avaliação da Feature Engineering 1: Qtde_Servicos

In [ ]:
# 4. Refaz o Split e o Pipeline usando os dados e colunas isolados
print("Refazendo Split e Pre-processamento para FE1...")
df_train_fe1, df_val_fe1, df_test_fe1 = realizar_split(df_fe1)

X_train_fe1, y_train_fe1, X_val_fe1, y_val_fe1, X_test_fe1, y_test_fe1 = aplicar_pipeline_e_oversampling_gb(
    df_train_fe1, df_val_fe1, df_test_fe1, cols_cat_fe1, cols_num_fe1
)
print(f"Tamanho de features de treino (FE1): {X_train_fe1.shape[1]} colunas.")

# --- PARTE 1: Experimento com os melhores parâmetros manuais (Bateria 6) ---
print("\nTreinando modelo de Ablação com os parâmetros do Campeão Manual (B6)...")
resultado_fe1_manual = executar_experimento_gb(
    name_experiment="GB_FE1_Qtde_Servicos_BaseB6",
    learning_rate=float(vencedor_b6['learning_rate']),
    n_estimators=int(vencedor_b6['n_estimators']),
    max_depth=int(vencedor_b6['max_depth']),
    min_samples_split=int(vencedor_b6['min_samples_split']),
    min_samples_leaf=int(vencedor_b6['min_samples_leaf']),
    subsample=float(vencedor_b6['subsample']),
    max_features=vencedor_b6['max_features'],
    X_train=X_train_fe1, y_train=y_train_fe1, X_val=X_val_fe1, y_val=y_val_fe1
)
_ = plotar_vencedor_gb([resultado_fe1_manual], X_val_fe1, y_val_fe1)

print("\n" + "="*50)
print(f"KS Anterior (Campeão B6 sem feature): {ks_b6_base:.4f}")
print(f"KS Atual (Campeão B6 com feature): {resultado_fe1_manual['val_ks']:.4f}")
print("="*50)

# --- PARTE 2: Experimento com os melhores parâmetros do Optuna Base ---
print("\n\nTreinando modelo de Ablação com os parâmetros do Campeão do Optuna Base...")
resultado_fe1_optuna = executar_experimento_gb(
    name_experiment="GB_FE1_Qtde_Servicos_BaseOptuna",
    learning_rate=best_trial_gb.params['learning_rate'],
    n_estimators=best_trial_gb.params['n_estimators'],
    max_depth=best_trial_gb.params['max_depth'],
    min_samples_split=best_trial_gb.params['min_samples_split'],
    min_samples_leaf=best_trial_gb.params['min_samples_leaf'],
    subsample=best_trial_gb.params['subsample'],
    max_features=best_trial_gb.params['max_features'],
    X_train=X_train_fe1, y_train=y_train_fe1, X_val=X_val_fe1, y_val=y_val_fe1
)
_ = plotar_vencedor_gb([resultado_fe1_optuna], X_val_fe1, y_val_fe1)

print("\n" + "="*50)
print(f"KS Anterior (Campeão Optuna sem feature): {ks_optuna_base_gb:.4f}")
print(f"KS Atual (Campeão Optuna com feature): {resultado_fe1_optuna['val_ks']:.4f}")
print("="*50)

### Comparação Visual: Qtde_Servicos

In [ ]:
plot_feature_engineering_comparison_gb(
    name_fe='Qtde_Servicos',
    ks_b6_base=ks_b6_base,
    ks_b6_fe=resultado_fe1_manual['val_ks'],
    ks_optuna_base=ks_optuna_base_gb,
    ks_optuna_fe=resultado_fe1_optuna['val_ks']
)

### Optuna FE_1 (Qtde_Servicos)

In [ ]:
study_fe1_gb = rodar_optuna_e_avaliar_gb(
    nome_estudo="Otimizacao_GB_FE1",
    X_train=X_train_fe1,
    y_train=y_train_fe1,
    X_val=X_val_fe1,
    y_val=y_val_fe1,
    prefixo_trial="Optuna_GB_FE1_Trial"
)

##FE_2: Tenure Binning (6 em 6 meses)

In [ ]:
# 1. Puxa NOVAMENTE a cópia da base virgem
df_fe2 = df_base.copy()
cols_cat_fe2 = colunas_categoricas_base.copy()
cols_num_fe2 = colunas_numericas_base.copy()

# 2. Aplica a transformação (Binning com include_lowest=True para incluir tenure=0)
bins = [0, 6, 12, 18, 24, 30, 78]
labels = ['0-6m', '7-12m', '13-18m', '19-24m', '25-30m', '30+m']
df_fe2['Tenure_Group'] = pd.cut(df_fe2['tenure'], bins=bins, labels=labels, right=True, include_lowest=True)

# 3. Adiciona APENAS na lista temporária categórica deste escopo
if 'Tenure_Group' not in cols_cat_fe2:
    cols_cat_fe2.append('Tenure_Group')

print("Feature 'Tenure_Group' criada (clientes com tenure=0 tratados corretamente)!")
display(df_fe2[['tenure', 'Tenure_Group']].head(10))

# 4. Refaz o Split e o Pipeline
print("Refazendo Split e Pre-processamento para FE2...")
df_train_fe2, df_val_fe2, df_test_fe2 = realizar_split(df_fe2)

X_train_fe2, y_train_fe2, X_val_fe2, y_val_fe2, X_test_fe2, y_test_fe2 = aplicar_pipeline_e_oversampling_gb(
    df_train_fe2, df_val_fe2, df_test_fe2, cols_cat_fe2, cols_num_fe2
)
print(f"Tamanho de features de treino (FE2): {X_train_fe2.shape[1]} colunas.")

# --- Experimentos de Ablação ---
print("\nTreinando modelo de Ablação com os parâmetros do Campeão Manual (B6)...")
resultado_fe2_manual = executar_experimento_gb(
    name_experiment="GB_FE2_TenureBin_BaseB6",
    learning_rate=float(vencedor_b6['learning_rate']),
    n_estimators=int(vencedor_b6['n_estimators']),
    max_depth=int(vencedor_b6['max_depth']),
    min_samples_split=int(vencedor_b6['min_samples_split']),
    min_samples_leaf=int(vencedor_b6['min_samples_leaf']),
    subsample=float(vencedor_b6['subsample']),
    max_features=vencedor_b6['max_features'],
    X_train=X_train_fe2, y_train=y_train_fe2, X_val=X_val_fe2, y_val=y_val_fe2
)
_ = plotar_vencedor_gb([resultado_fe2_manual], X_val_fe2, y_val_fe2)

print("\n" + "="*50)
print(f"KS Anterior (Campeão B6 sem Tenure_Group): {ks_b6_base:.4f}")
print(f"KS Atual (Campeão B6 com Tenure_Group): {resultado_fe2_manual['val_ks']:.4f}")
print("="*50)

print("\n\nTreinando modelo de Ablação com os parâmetros do Campeão do Optuna Base...")
resultado_fe2_optuna = executar_experimento_gb(
    name_experiment="GB_FE2_TenureBin_BaseOptuna",
    learning_rate=best_trial_gb.params['learning_rate'],
    n_estimators=best_trial_gb.params['n_estimators'],
    max_depth=best_trial_gb.params['max_depth'],
    min_samples_split=best_trial_gb.params['min_samples_split'],
    min_samples_leaf=best_trial_gb.params['min_samples_leaf'],
    subsample=best_trial_gb.params['subsample'],
    max_features=best_trial_gb.params['max_features'],
    X_train=X_train_fe2, y_train=y_train_fe2, X_val=X_val_fe2, y_val=y_val_fe2
)
_ = plotar_vencedor_gb([resultado_fe2_optuna], X_val_fe2, y_val_fe2)

print("\n" + "="*50)
print(f"KS Anterior (Campeão Optuna sem Tenure_Group): {ks_optuna_base_gb:.4f}")
print(f"KS Atual (Campeão Optuna com Tenure_Group): {resultado_fe2_optuna['val_ks']:.4f}")
print("="*50)

### Comparação Visual: Tenure_Group

In [ ]:
plot_feature_engineering_comparison_gb(
    name_fe='Tenure_Group (Cumulativo)',
    ks_b6_base=ks_b6_base,
    ks_b6_fe=resultado_fe2_manual['val_ks'],
    ks_optuna_base=ks_optuna_base_gb,
    ks_optuna_fe=resultado_fe2_optuna['val_ks']
)

### Optuna FE_2 (Tenure Binning)

In [ ]:
study_fe2_gb = rodar_optuna_e_avaliar_gb(
    nome_estudo="Otimizacao_GB_FE2",
    X_train=X_train_fe2,
    y_train=y_train_fe2,
    X_val=X_val_fe2,
    y_val=y_val_fe2,
    prefixo_trial="Optuna_GB_FE2_Trial"
)

## FE_3: Proporção de Gasto (Gasto_Medio_Real)

In [ ]:
# 1. Puxa a cópia da base virgem e das listas
df_fe3 = df_base.copy()
cols_cat_fe3 = colunas_categoricas_base.copy()
cols_num_fe3 = colunas_numericas_base.copy()

# 2. Aplica a transformação (Evitando divisão por zero onde tenure == 0)
df_fe3['Gasto_Medio_Real'] = np.where(df_fe3['tenure'] == 0, 0, df_fe3['TotalCharges'] / df_fe3['tenure'])

# 3. Adiciona APENAS na lista temporária deste escopo
if 'Gasto_Medio_Real' not in cols_num_fe3:
    cols_num_fe3.append('Gasto_Medio_Real')

print("Feature 'Gasto_Medio_Real' criada e adicionada às colunas numéricas!")
display(df_fe3[['TotalCharges', 'tenure', 'Gasto_Medio_Real', 'MonthlyCharges']].head(5))

# 4. Refaz o Split e o Pipeline
print("Refazendo Split e Pre-processamento para FE3...")
df_train_fe3, df_val_fe3, df_test_fe3 = realizar_split(df_fe3)

X_train_fe3, y_train_fe3, X_val_fe3, y_val_fe3, X_test_fe3, y_test_fe3 = aplicar_pipeline_e_oversampling_gb(
    df_train_fe3, df_val_fe3, df_test_fe3, cols_cat_fe3, cols_num_fe3
)
print(f"Tamanho de features de treino (FE3): {X_train_fe3.shape[1]} colunas.")

# --- Experimentos de Ablação ---
print("\nTreinando modelo de Ablação com os parâmetros do Campeão Manual (B6)...")
resultado_fe3_manual = executar_experimento_gb(
    name_experiment="GB_FE3_GastoMedio_BaseB6",
    learning_rate=float(vencedor_b6['learning_rate']),
    n_estimators=int(vencedor_b6['n_estimators']),
    max_depth=int(vencedor_b6['max_depth']),
    min_samples_split=int(vencedor_b6['min_samples_split']),
    min_samples_leaf=int(vencedor_b6['min_samples_leaf']),
    subsample=float(vencedor_b6['subsample']),
    max_features=vencedor_b6['max_features'],
    X_train=X_train_fe3, y_train=y_train_fe3, X_val=X_val_fe3, y_val=y_val_fe3
)
_ = plotar_vencedor_gb([resultado_fe3_manual], X_val_fe3, y_val_fe3)

print("\n" + "="*50)
print(f"KS Anterior (Campeão B6 sem Gasto_Medio): {ks_b6_base:.4f}")
print(f"KS Atual (Campeão B6 com Gasto_Medio): {resultado_fe3_manual['val_ks']:.4f}")
print("="*50)

print("\n\nTreinando modelo de Ablação com os parâmetros do Campeão do Optuna Base...")
resultado_fe3_optuna = executar_experimento_gb(
    name_experiment="GB_FE3_GastoMedio_BaseOptuna",
    learning_rate=best_trial_gb.params['learning_rate'],
    n_estimators=best_trial_gb.params['n_estimators'],
    max_depth=best_trial_gb.params['max_depth'],
    min_samples_split=best_trial_gb.params['min_samples_split'],
    min_samples_leaf=best_trial_gb.params['min_samples_leaf'],
    subsample=best_trial_gb.params['subsample'],
    max_features=best_trial_gb.params['max_features'],
    X_train=X_train_fe3, y_train=y_train_fe3, X_val=X_val_fe3, y_val=y_val_fe3
)
_ = plotar_vencedor_gb([resultado_fe3_optuna], X_val_fe3, y_val_fe3)

print("\n" + "="*50)
print(f"KS Anterior (Campeão Optuna sem Gasto_Medio): {ks_optuna_base_gb:.4f}")
print(f"KS Atual (Campeão Optuna com Gasto_Medio): {resultado_fe3_optuna['val_ks']:.4f}")
print("="*50)

### Comparação Visual: Proporção de Gasto

In [ ]:
plot_feature_engineering_comparison_gb(
    name_fe='Proporção de Gasto',
    ks_b6_base=ks_b6_base,
    ks_b6_fe=resultado_fe3_manual['val_ks'],
    ks_optuna_base=ks_optuna_base_gb,
    ks_optuna_fe=resultado_fe3_optuna['val_ks']
)

### Optuna FE_3 (Proporção de Gasto)

In [ ]:
study_fe3_gb = rodar_optuna_e_avaliar_gb(
    nome_estudo="Otimizacao_GB_FE3",
    X_train=X_train_fe3,
    y_train=y_train_fe3,
    X_val=X_val_fe3,
    y_val=y_val_fe3,
    prefixo_trial="Optuna_GB_FE3_Trial"
)

## FE_4: Família (Partner + Dependents)

In [ ]:
# 1. Puxa a cópia da base virgem e das listas
df_fe4 = df_base.copy()
cols_cat_fe4 = colunas_categoricas_base.copy()
cols_num_fe4 = colunas_numericas_base.copy()

# 2. Aplica a transformação
df_fe4['Tem_Familia'] = ((df_fe4['Partner'] == 'Yes') | (df_fe4['Dependents'] == 'Yes')).map({True: 'Yes', False: 'No'})

# 3. Adiciona APENAS na lista temporária deste escopo
if 'Tem_Familia' not in cols_cat_fe4:
    cols_cat_fe4.append('Tem_Familia')

print("Feature 'Tem_Familia' criada e adicionada às colunas categóricas!")
display(df_fe4[['Partner', 'Dependents', 'Tem_Familia']].head(5))

# 4. Refaz o Split e o Pipeline
print("Refazendo Split e Pre-processamento para FE4...")
df_train_fe4, df_val_fe4, df_test_fe4 = realizar_split(df_fe4)

X_train_fe4, y_train_fe4, X_val_fe4, y_val_fe4, X_test_fe4, y_test_fe4 = aplicar_pipeline_e_oversampling_gb(
    df_train_fe4, df_val_fe4, df_test_fe4, cols_cat_fe4, cols_num_fe4
)
print(f"Tamanho de features de treino (FE4): {X_train_fe4.shape[1]} colunas.")

# --- Experimentos de Ablação ---
print("\nTreinando modelo de Ablação com os parâmetros do Campeão Manual (B6)...")
resultado_fe4_manual = executar_experimento_gb(
    name_experiment="GB_FE4_Familia_BaseB6",
    learning_rate=float(vencedor_b6['learning_rate']),
    n_estimators=int(vencedor_b6['n_estimators']),
    max_depth=int(vencedor_b6['max_depth']),
    min_samples_split=int(vencedor_b6['min_samples_split']),
    min_samples_leaf=int(vencedor_b6['min_samples_leaf']),
    subsample=float(vencedor_b6['subsample']),
    max_features=vencedor_b6['max_features'],
    X_train=X_train_fe4, y_train=y_train_fe4, X_val=X_val_fe4, y_val=y_val_fe4
)
_ = plotar_vencedor_gb([resultado_fe4_manual], X_val_fe4, y_val_fe4)

print("\n" + "="*50)
print(f"KS Anterior (Campeão B6 sem Familia): {ks_b6_base:.4f}")
print(f"KS Atual (Campeão B6 com Familia): {resultado_fe4_manual['val_ks']:.4f}")
print("="*50)

print("\n\nTreinando modelo de Ablação com os parâmetros do Campeão do Optuna Base...")
resultado_fe4_optuna = executar_experimento_gb(
    name_experiment="GB_FE4_Familia_BaseOptuna",
    learning_rate=best_trial_gb.params['learning_rate'],
    n_estimators=best_trial_gb.params['n_estimators'],
    max_depth=best_trial_gb.params['max_depth'],
    min_samples_split=best_trial_gb.params['min_samples_split'],
    min_samples_leaf=best_trial_gb.params['min_samples_leaf'],
    subsample=best_trial_gb.params['subsample'],
    max_features=best_trial_gb.params['max_features'],
    X_train=X_train_fe4, y_train=y_train_fe4, X_val=X_val_fe4, y_val=y_val_fe4
)
_ = plotar_vencedor_gb([resultado_fe4_optuna], X_val_fe4, y_val_fe4)

print("\n" + "="*50)
print(f"KS Anterior (Campeão Optuna sem Familia): {ks_optuna_base_gb:.4f}")
print(f"KS Atual (Campeão Optuna com Familia): {resultado_fe4_optuna['val_ks']:.4f}")
print("="*50)

### Comparação Visual: Família

In [ ]:
plot_feature_engineering_comparison_gb(
    name_fe='Família',
    ks_b6_base=ks_b6_base,
    ks_b6_fe=resultado_fe4_manual['val_ks'],
    ks_optuna_base=ks_optuna_base_gb,
    ks_optuna_fe=resultado_fe4_optuna['val_ks']
)

### Optuna FE_4 (Família)

In [ ]:
study_fe4_gb = rodar_optuna_e_avaliar_gb(
    nome_estudo="Otimizacao_GB_FE4",
    X_train=X_train_fe4,
    y_train=y_train_fe4,
    X_val=X_val_fe4,
    y_val=y_val_fe4,
    prefixo_trial="Optuna_GB_FE4_Trial"
)

## FE_ALL: Todas as Features Combinadas + Optuna

In [ ]:
# 1. Base virgem
df_fe_all = df_base.copy()
cols_cat_all = colunas_categoricas_base.copy()
cols_num_all = colunas_numericas_base.copy()

# FE_1: Qtde_Servicos
colunas_servicos = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_fe_all['Qtde_Servicos'] = df_fe_all[colunas_servicos].apply(lambda linha: (linha == 'Yes').sum(), axis=1)
cols_num_all.append('Qtde_Servicos')

# FE_2: Tenure Binning (Corrigido para tenure = 0)
bins = [0, 6, 12, 18, 24, 30, 78]
labels = ['0-6m', '7-12m', '13-18m', '19-24m', '25-30m', '30+m']
df_fe_all['Tenure_Group'] = pd.cut(df_fe_all['tenure'], bins=bins, labels=labels, right=True, include_lowest=True)
cols_cat_all.append('Tenure_Group')

# FE_3: Proporção de Gasto
df_fe_all['Gasto_Medio_Real'] = np.where(df_fe_all['tenure'] == 0, 0, df_fe_all['TotalCharges'] / df_fe_all['tenure'])
cols_num_all.append('Gasto_Medio_Real')

# FE_4: Família
df_fe_all['Tem_Familia'] = ((df_fe_all['Partner'] == 'Yes') | (df_fe_all['Dependents'] == 'Yes')).map({True: 'Yes', False: 'No'})
cols_cat_all.append('Tem_Familia')

# --- SPLIT E PRE-PROCESSAMENTO ---
print("Refazendo Split e Pre-processamento para FE_ALL...")
df_train_all, df_val_all, df_test_all = realizar_split(df_fe_all)

X_train_all, y_train_all, X_val_all, y_val_all, X_test_all, y_test_all = aplicar_pipeline_e_oversampling_gb(
    df_train_all, df_val_all, df_test_all, cols_cat_all, cols_num_all
)
print(f"Tamanho de features de treino (Todas as FEs): {X_train_all.shape[1]} colunas.")
display(X_train_all.head(3))

# --- Experimentos de Ablação ---
print("\nTreinando modelo de Ablação com os parâmetros do Campeão Manual (B6)...")
resultado_fe_all_manual = executar_experimento_gb(
    name_experiment="GB_FE_ALL_BaseB6",
    learning_rate=float(vencedor_b6['learning_rate']),
    n_estimators=int(vencedor_b6['n_estimators']),
    max_depth=int(vencedor_b6['max_depth']),
    min_samples_split=int(vencedor_b6['min_samples_split']),
    min_samples_leaf=int(vencedor_b6['min_samples_leaf']),
    subsample=float(vencedor_b6['subsample']),
    max_features=vencedor_b6['max_features'],
    X_train=X_train_all, y_train=y_train_all, X_val=X_val_all, y_val=y_val_all
)
_ = plotar_vencedor_gb([resultado_fe_all_manual], X_val_all, y_val_all)

print("\n" + "="*50)
print(f"KS Anterior (Campeão B6 sem FE): {ks_b6_base:.4f}")
print(f"KS Atual (Campeão B6 com FE_ALL): {resultado_fe_all_manual['val_ks']:.4f}")
print("="*50)

print("\n\nTreinando modelo de Ablação com os parâmetros do Campeão do Optuna Base...")
resultado_fe_all_optuna = executar_experimento_gb(
    name_experiment="GB_FE_ALL_BaseOptuna",
    learning_rate=best_trial_gb.params['learning_rate'],
    n_estimators=best_trial_gb.params['n_estimators'],
    max_depth=best_trial_gb.params['max_depth'],
    min_samples_split=best_trial_gb.params['min_samples_split'],
    min_samples_leaf=best_trial_gb.params['min_samples_leaf'],
    subsample=best_trial_gb.params['subsample'],
    max_features=best_trial_gb.params['max_features'],
    X_train=X_train_all, y_train=y_train_all, X_val=X_val_all, y_val=y_val_all
)
_ = plotar_vencedor_gb([resultado_fe_all_optuna], X_val_all, y_val_all)

print("\n" + "="*50)
print(f"KS Anterior (Campeão Optuna sem FE): {ks_optuna_base_gb:.4f}")
print(f"KS Atual (Campeão Optuna com FE_ALL): {resultado_fe_all_optuna['val_ks']:.4f}")
print("="*50)

### Comparação Visual: Todas as Features (FE_ALL)

In [ ]:
plot_feature_engineering_comparison_gb(
    name_fe='Todas as Features (FE_ALL)',
    ks_b6_base=ks_b6_base,
    ks_b6_fe=resultado_fe_all_manual['val_ks'],
    ks_optuna_base=ks_optuna_base_gb,
    ks_optuna_fe=resultado_fe_all_optuna['val_ks']
)

###Optuna FE_ALL

In [ ]:
study_all_gb = rodar_optuna_e_avaliar_gb(
    nome_estudo="Otimizacao_GB_FE_ALL",
    X_train=X_train_all,
    y_train=y_train_all,
    X_val=X_val_all,
    y_val=y_val_all,
    prefixo_trial="Optuna_GB_FE_ALL_Trial"
)

## Ranking Final: Seleção do Modelo Definitivo
Aqui nós consolidamos todos os estudos realizados para comparar qual foi o Grande Campeão entre o modelo Base e todas as estratégias de Feature Engineering.

In [ ]:
import pandas as pd
import optuna
import os

def consolidar_resultados_optuna_gb(caminho_logs_gb):
    estudos = [
        ("Base (Sem FE)", "Otimizacao_GB", "optuna_gb_baseline.db"),
        ("FE 1 (Qtde Serviços)", "Otimizacao_GB_FE1", "otimizacao_gb_fe1.db"),
        ("FE 2 (Tenure Binning)", "Otimizacao_GB_FE2", "otimizacao_gb_fe2.db"),
        ("FE 3 (Proporção Gasto)", "Otimizacao_GB_FE3", "otimizacao_gb_fe3.db"),
        ("FE 4 (Família)", "Otimizacao_GB_FE4", "otimizacao_gb_fe4.db"),
        ("FE ALL (Todas as FEs)", "Otimizacao_GB_FE_ALL", "otimizacao_gb_fe_all.db")
    ]

    resultados = []

    # 1. Carrega o campeão dos testes Manuais (Bateria 6)
    try:
        caminho_csv_b6 = os.path.join(caminho_logs_gb, "bateria_6_max_features.csv")
        if os.path.exists(caminho_csv_b6):
            df_b6 = pd.read_csv(caminho_csv_b6)
            melhor_linha_b6 = df_b6.loc[df_b6['val_ks'].idxmax()]

            linha_b6 = {
                "Experimento": "Manual Campeão (Bateria 6)",
                "Melhor KS": melhor_linha_b6['val_ks'],
                "Trial ID Vencedor": melhor_linha_b6['nome'],
                "learning_rate": melhor_linha_b6['learning_rate'],
                "n_estimators": melhor_linha_b6['n_estimators'],
                "max_depth": melhor_linha_b6['max_depth'],
                "min_samples_split": melhor_linha_b6['min_samples_split'],
                "min_samples_leaf": melhor_linha_b6['min_samples_leaf'],
                "subsample": melhor_linha_b6['subsample'],
                "max_features": melhor_linha_b6['max_features']
            }
            resultados.append(linha_b6)
    except Exception as e:
        print(f"Aviso: Não foi possível carregar o estudo Manual da Bateria 6 - {e}")

    # 2. Carrega os estudos do Optuna
    for nome_amigavel, nome_estudo, arquivo_db in estudos:
        caminho_banco = f"sqlite:///{caminho_logs_gb}/{arquivo_db}"
        try:
            study = optuna.load_study(study_name=nome_estudo, storage=caminho_banco)
            best_trial = study.best_trial

            linha = {
                "Experimento": nome_amigavel,
                "Melhor KS": best_trial.value,
                "Trial ID Vencedor": best_trial.number
            }
            linha.update(best_trial.params.copy())
            resultados.append(linha)
        except Exception as e:
            print(f"Aviso: Não foi possível carregar o estudo '{nome_estudo}' - {e}")

    df_ranking = pd.DataFrame(resultados)

    if not df_ranking.empty:
        df_ranking = df_ranking.sort_values(by="Melhor KS", ascending=False).reset_index(drop=True)
        caminho_csv = os.path.join(caminho_logs_gb, "ranking_final_modelos_gb.csv")
        df_ranking.to_csv(caminho_csv, index=False)
        print(f"Ranking consolidado salvo com sucesso em: {caminho_csv}\n")

    return df_ranking

df_ranking_final_gb = consolidar_resultados_optuna_gb(caminho_logs_gb)

print("=== PÓDIO DOS MODELOS (GRADIENT BOOSTING) ===")
display(df_ranking_final_gb.head(10))

# Avaliação Final no Conjunto de Teste
Agora vamos pegar o primeiro colocado do nosso ranking, carregar o modelo salvo e avaliar seu poder de predição nos dados de teste (`X_test` e `y_test`).

In [ ]:
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scikitplot as skplt
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay

# 1. Pega as informações do Melhor Modelo Geral
melhor_modelo_info = df_ranking_final_gb.iloc[0]
experimento_campeao = melhor_modelo_info['Experimento']
trial_id = melhor_modelo_info['Trial ID Vencedor']

# 2. Dicionário para mapear qual Experimento usa qual Dataset de Teste/Validação e qual o prefixo do arquivo salvo
mapa_experimentos_gb = {
    "Manual Campeão (Bateria 6)": {"X_train": X_train_gb, "y_train": y_train_gb, "X_test": X_test_gb, "X_val": X_val_gb, "y_val": y_val_gb, "y_test": y_test_gb, "prefixo": ""},
    "Base (Sem FE)": {"X_train": X_train_gb, "y_train": y_train_gb, "X_test": X_test_gb, "X_val": X_val_gb, "y_val": y_val_gb, "y_test": y_test_gb, "prefixo": "Optuna_Trial_"},
    "FE 1 (Qtde Serviços)": {"X_train": X_train_fe1, "y_train": y_train_fe1, "X_test": X_test_fe1, "X_val": X_val_fe1, "y_val": y_val_fe1, "y_test": y_test_fe1, "prefixo": "Optuna_GB_FE1_Trial_"},
    "FE 2 (Tenure Binning)": {"X_train": X_train_fe2, "y_train": y_train_fe2, "X_test": X_test_fe2, "X_val": X_val_fe2, "y_val": y_val_fe2, "y_test": y_test_fe2, "prefixo": "Optuna_GB_FE2_Trial_"},
    "FE 3 (Proporção Gasto)": {"X_train": X_train_fe3, "y_train": y_train_fe3, "X_test": X_test_fe3, "X_val": X_val_fe3, "y_val": y_val_fe3, "y_test": y_test_fe3, "prefixo": "Optuna_GB_FE3_Trial_"},
    "FE 4 (Família)": {"X_train": X_train_fe4, "y_train": y_train_fe4, "X_test": X_test_fe4, "X_val": X_val_fe4, "y_val": y_val_fe4, "y_test": y_test_fe4, "prefixo": "Optuna_GB_FE4_Trial_"},
    "FE ALL (Todas as FEs)": {"X_train": X_train_all, "y_train": y_train_all, "X_test": X_test_all, "X_val": X_val_all, "y_val": y_val_all, "y_test": y_test_all, "prefixo": "Optuna_GB_FE_ALL_Trial_"}
}

config_vencedor = mapa_experimentos_gb[experimento_campeao]
X_train_campeao = config_vencedor["X_train"]
y_train_campeao = config_vencedor["y_train"]
X_test_campeao = config_vencedor["X_test"]
X_val_campeao = config_vencedor["X_val"]
y_val_campeao = config_vencedor["y_val"]
y_test_campeao = config_vencedor["y_test"]

if isinstance(trial_id, (int, np.integer)) or str(trial_id).isdigit():
    nome_modelo_salvo = f"{config_vencedor['prefixo']}{int(trial_id)}"
else:
    nome_modelo_salvo = str(trial_id)

print(f"O CAMPEÃO GERAL FOI O EXPERIMENTO: '{experimento_campeao}'")
print(f"Carregando modelo: {nome_modelo_salvo}.joblib")

print("\n" + "*"*60)
print("AVALIAÇÃO DEFINITIVA NO CONJUNTO DE TESTE (UNSEEN DATA)")
print("*"*60)

# 3. Carregando o modelo campeão do Drive
caminho_arquivo = os.path.join(caminho_modelos_gb, f"{nome_modelo_salvo}.joblib")
modelo_campeao = joblib.load(caminho_arquivo)

# ---------------------------------------------------------
# 3.5. Avaliando no conjunto de TREINO (para comparação de MSE/Generalização - Slide 14)
# ---------------------------------------------------------
probs_1_train = modelo_campeao.predict_proba(X_train_campeao)[:, 1]
y_true_train = np.array(y_train_campeao)
scores_train = get_scores_churn(y_true_train, probs_1_train)

# ---------------------------------------------------------
# 4. Obtendo o Threshold ótimo usando APENAS a VALIDAÇÃO
# ---------------------------------------------------------
probs_1_val = modelo_campeao.predict_proba(X_val_campeao)[:, 1]
y_true_val = np.array(y_val_campeao)

scores_val = get_scores_churn(y_true_val, probs_1_val)
optimal_thresh = scores_val['optimal_threshold']
print(f"Threshold ótimo obtido via Validação: {optimal_thresh:.3f}\n")

# ---------------------------------------------------------
# 5. Avalia no conjunto de Teste
# ---------------------------------------------------------
probs_1_test = modelo_campeao.predict_proba(X_test_campeao)[:, 1]
y_true_test = np.array(y_test_campeao)
probs_0_test = 1 - probs_1_test
y_pred_scores = np.column_stack((probs_0_test, probs_1_test))

scores_default_test = get_scores_churn(y_true_test, probs_1_test, threshold=0.5)
scores_optimal_test = get_scores_churn(y_true_test, probs_1_test, threshold=optimal_thresh)

y_pred_default = (probs_1_test >= 0.5).astype(int)
y_pred_otimizado = (probs_1_test >= optimal_thresh).astype(int)

# --- TABELA E GRÁFICO DE GENERALIZAÇÃO: MSE, LOG-LOSS, KS E AUROC (TREINO vs VALIDAÇÃO vs TESTE) ---
df_generalizacao = pd.DataFrame([
    {"Conjunto": "Treino",    "KS": scores_train['ks_stat'],        "AUROC": scores_train['roc_auc'],        "MSE (Brier)": scores_train['mse'],        "Cross-Entropy (Log-Loss)": scores_train['log_loss']},
    {"Conjunto": "Validação", "KS": scores_val['ks_stat'],          "AUROC": scores_val['roc_auc'],          "MSE (Brier)": scores_val['mse'],          "Cross-Entropy (Log-Loss)": scores_val['log_loss']},
    {"Conjunto": "Teste",     "KS": scores_default_test['ks_stat'], "AUROC": scores_default_test['roc_auc'], "MSE (Brier)": scores_default_test['mse'], "Cross-Entropy (Log-Loss)": scores_default_test['log_loss']}
])
print("--- COMPARATIVO DE GENERALIZAÇÃO (TREINO vs VALIDAÇÃO vs TESTE) ---")
display(df_generalizacao.round(4))

fig_mse, axes_mse = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=df_generalizacao, x="Conjunto", y="MSE (Brier)", hue="Conjunto",
            palette=["#4C72B0", "#DD8452", "#55A868"], legend=False, ax=axes_mse[0])
for p in axes_mse[0].patches:
    axes_mse[0].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 8), textcoords='offset points', fontweight='bold')
axes_mse[0].set_ylim(0, df_generalizacao["MSE (Brier)"].max() * 1.18)
axes_mse[0].set_title(f"MSE (Brier Score) - {nome_modelo_salvo}")
axes_mse[0].set_ylabel("MSE (Menor é Melhor)")
axes_mse[0].grid(axis='y', linestyle='--', alpha=0.6)

sns.barplot(data=df_generalizacao, x="Conjunto", y="KS", hue="Conjunto",
            palette=["#4C72B0", "#DD8452", "#55A868"], legend=False, ax=axes_mse[1])
for p in axes_mse[1].patches:
    axes_mse[1].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 8), textcoords='offset points', fontweight='bold')
axes_mse[1].set_ylim(0, df_generalizacao["KS"].max() * 1.18)
axes_mse[1].set_title(f"KS - {nome_modelo_salvo}")
axes_mse[1].set_ylabel("KS (Maior é Melhor)")
axes_mse[1].grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# --- PRINT DOS RELATÓRIOS ---
print(f"--- MÉTRICAS NO TESTE COM THRESHOLD PADRÃO (0.500) ---")
print(f"KS: {scores_default_test['ks_stat']:.4f} | AUROC: {scores_default_test['roc_auc']:.4f} | MSE: {scores_default_test['mse']:.4f} | Log-Loss: {scores_default_test['log_loss']:.4f} | Acurácia: {scores_default_test['accuracy']:.4f}")
print(f"Precisão: {scores_default_test['precision']:.4f} | Recall: {scores_default_test['recall']:.4f} | F1-Score: {scores_default_test['f1_score']:.4f}\n")
print(classification_report(y_true_test, y_pred_default, target_names=['Não Churn (0)', 'Churn (1)']))

print(f"\n--- MÉTRICAS NO TESTE COM THRESHOLD OTIMIZADO ({optimal_thresh:.3f}) ---")
print(f"KS: {scores_optimal_test['ks_stat']:.4f} | AUROC: {scores_optimal_test['roc_auc']:.4f} | Acurácia: {scores_optimal_test['accuracy']:.4f}")
print(f"Precisão: {scores_optimal_test['precision']:.4f} | Recall: {scores_optimal_test['recall']:.4f} | F1-Score: {scores_optimal_test['f1_score']:.4f}\n")
print(classification_report(y_true_test, y_pred_otimizado, target_names=['Não Churn (0)', 'Churn (1)']))

# --- PLOTAGEM DOS GRÁFICOS E MATRIZES EM PORCENTAGEM ---
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

skplt.metrics.plot_ks_statistic(y_true_test, y_pred_scores, ax=axes[0], title=f"KS: {nome_modelo_salvo} (Teste)")

RocCurveDisplay.from_predictions(y_true_test, probs_1_test, ax=axes[1], name=nome_modelo_salvo)
axes[1].set_title("Curva ROC (Teste)")
axes[1].plot([0, 1], [0, 1], 'k--', label='Aleatório')
axes[1].legend()

cm_norm_default = confusion_matrix(y_true_test, y_pred_default, normalize='true')
sns.heatmap(cm_norm_default, annot=True, fmt=".2%", cmap="Blues", ax=axes[2],
            xticklabels=['Não Churn (0)', 'Churn (1)'],
            yticklabels=['Não Churn (0)', 'Churn (1)'])
axes[2].set_title("Matriz de Confusão % (Thr: 0.500)")
axes[2].set_ylabel("Valor Real")
axes[2].set_xlabel("Predição")

cm_norm_optimal = confusion_matrix(y_true_test, y_pred_otimizado, normalize='true')
sns.heatmap(cm_norm_optimal, annot=True, fmt=".2%", cmap="Greens", ax=axes[3],
            xticklabels=['Não Churn (0)', 'Churn (1)'],
            yticklabels=['Não Churn (0)', 'Churn (1)'])
axes[3].set_title(f"Matriz de Confusão % (Thr KS: {optimal_thresh:.3f})")
axes[3].set_ylabel("Valor Real")
axes[3].set_xlabel("Predição")

plt.tight_layout()
plt.show()

# --- PLOTAGEM DAS MATRIZES ABSOLUTAS ---
cm_abs_default = confusion_matrix(y_true_test, y_pred_default)
cm_abs_optimal = confusion_matrix(y_true_test, y_pred_otimizado)

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_abs_default, annot=True, fmt="d", cmap="Blues", ax=axes2[0],
            xticklabels=['Não Churn (0)', 'Churn (1)'],
            yticklabels=['Não Churn (0)', 'Churn (1)'])
axes2[0].set_title("Matriz de Confusão Absoluta (Thr: 0.500)")
axes2[0].set_ylabel("Valor Real")
axes2[0].set_xlabel("Predição")

sns.heatmap(cm_abs_optimal, annot=True, fmt="d", cmap="Oranges", ax=axes2[1],
            xticklabels=['Não Churn (0)', 'Churn (1)'],
            yticklabels=['Não Churn (0)', 'Churn (1)'])
axes2[1].set_title(f"Matriz de Confusão Absoluta (Thr KS: {optimal_thresh:.3f})")
axes2[1].set_ylabel("Valor Real")
axes2[1].set_xlabel("Predição")

plt.tight_layout()
plt.show()